# Judge-vs-Pathologist Agreement (Phase 1: Benchmarks 1-4)

Validates InternVL3.5-38B and Qwen3-VL-32B-Thinking as judges by comparing their
scores against the human pathologist ratings already collected, for every row the
pathologists rated (all 5 evaluators' assigned parts pooled, ~231 PathOPEN core
rows / 174 augmentation rows / 662 filtered-PathVQA rows).

**Important**: each row was rated by exactly one of the 5 pathologists (the parts
are disjoint, verified separately - zero row overlap between `evaluator1`..`evaluator5`
CSVs). So this is not classic multi-rater IRR; it is "judge score vs. whichever single
pathologist rated this row", pooled across the full dataset to get one agreement
statistic per (benchmark x criterion x dataset).

For each (benchmark, criterion, dataset) combination this notebook computes:
- **Weighted Cohen's kappa** (quadratic weights, appropriate for ordinal -1..2 data)
  between judge score and human score.
- **Mann-Whitney U test + rank-biserial effect size**, comparing the score
  *distribution* on PathOPEN vs. filtered-PathVQA (Pillar 1a's stated method),
  computed separately for human ratings and for each judge.

## `-1` is a score, not a missing value

Rows rated `-1` ("unable to comprehend the question and/or image, or unable to make
the evaluation") are **included** in both the kappa and the Mann-Whitney statistics.

`-1` is a defined level of the rubric (Tables 5-10), and a pathologist assigning it
is making a substantive judgment: *this item is defective*. That is precisely the
dataset-quality signal Pillar 1 is built on. An earlier version of this notebook
excluded rows where the human scored `-1`, which was wrong in two ways:

- It **deleted the finding**. The claim "PathOPEN contains fewer uninterpretable
  items than PathVQA" cannot be measured by a procedure that first removes every
  uninterpretable item.
- It **biased the comparison asymmetrically**. PathVQA carries far more `-1`s than
  PathOPEN, so the exclusion removed more rows from the dataset the paper argues is
  weaker, flattering it and understating the gap.

Whether a judge *reproduces* the pathologist's `-1` is itself a validation question,
and it can only be asked with `-1` in the data. The output tables now carry
`n_human_neg1` / `n_judge_neg1` / `n_both_neg1` so the contribution is visible rather
than buried in the aggregate.

**Caveat to report with these numbers**: quadratic weights treat the scale as evenly
spaced, so a `-1`-vs-`2` disagreement is penalized as a 3-step gap, more heavily than
`0`-vs-`2`. `-1` is arguably a different *kind* of judgment ("unscorable") rather than
a rung below `0`. The kappas are therefore conservative wherever the two raters
disagree about whether an item is scorable at all.

Note that `wrong_answer_tier_agreement.ipynb` still excludes `-1`, and correctly so:
it maps scores onto a 3-tier scale (near/moderate/far-miss) in which `-1` has no tier,
per the paper's own Pillar 2 definition. That is a structural property of the tiering,
not a filtering choice.

**Prerequisite**: run `judge_runner_pathopen.ipynb` and `judge_runner_pathvqa.ipynb`
first so `judge_output/evaluator_internvl/` and `judge_output/evaluator_qwenvl/`
exist.


In [ ]:
import glob
import os

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.metrics import cohen_kappa_score


In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
HUMAN_INPUT_DIR = os.path.join(
    REPO_ROOT, "data_evaluation", "pathologists", "scoring_analysis", "input"
)
JUDGE_OUTPUT_DIR = os.path.join(os.getcwd(), "judge_output")
JUDGE_KEYS = ["internvl", "qwenvl"]

REPO_ROOT, HUMAN_INPUT_DIR, JUDGE_OUTPUT_DIR


## Load and pool human ratings

Pools `evaluator1`..`evaluator5`'s files per dataset into one table (`__evaluator__`
column retained for traceability), dropping the benchmark-label header row that each
raw CSV carries at index 0 (same convention as the existing
`*_scoring_analysis_individual.ipynb` notebooks).

For PathOPEN, `Image_ID` is unique across the pooled set (verified separately - the
5 evaluators' files are disjoint row slices with no overlap), so it is a safe join
key on its own.

For **filtered PathVQA, `image_id` is NOT unique** (the same image hosts several
different questions - 224 duplicate `Image_ID` values across the pooled human
ratings). Each `evaluatorN/pathvqa_eval_data.csv` is row-for-row aligned with
`subsets_processing_output/data/pathvqa_partN.csv` (verified: identical question
text at every row position, per evaluator). So a global `row_uid` is reconstructed
here by concatenating `evaluator1`..`evaluator5` in that exact numeric order,
matching how `judge_runner_pathvqa.ipynb` builds its own `row_uid` by concatenating
`pathvqa_part1`..`part5` in sorted order.

In [ ]:
def _evaluator_dirs_in_order() -> list:
    """evaluator1..evaluator5 in strict numeric order (not lexicographic - avoids a
    latent evaluator10-before-evaluator2 bug if the evaluator count ever grows past 9)."""
    dirs = glob.glob(os.path.join(HUMAN_INPUT_DIR, "evaluator[0-9]*"))
    return sorted(dirs, key=lambda p: int(os.path.basename(p).replace("evaluator", "")))


def load_pooled_human(dataset_filename: str, add_row_uid: bool = False) -> pd.DataFrame:
    frames = []
    for evaluator_dir in _evaluator_dirs_in_order():
        path = os.path.join(evaluator_dir, dataset_filename)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        df = df.drop(index=0).reset_index(drop=True)  # drop benchmark-label header row
        df["__evaluator__"] = os.path.basename(evaluator_dir)
        frames.append(df)
    pooled = pd.concat(frames, ignore_index=True)
    if add_row_uid:
        pooled["row_uid"] = range(len(pooled))
    return pooled


human_pathopen = load_pooled_human("pathopen_eval_data.csv")
human_pathvqa = load_pooled_human("pathvqa_eval_data.csv", add_row_uid=True)
# Benchmark 4 (image augmentation) - Pillar 5a's validation evidence. Same pooled-CSV
# convention as the other two; Image_ID is unique here too (verified below), so it joins
# to the judge output directly.
human_augmentation = load_pooled_human("pathopen_image_augmentation_eval_data.csv")

assert human_augmentation["Image_ID"].is_unique, (
    "augmentation Image_ID is not unique across the pooled evaluators; the join key "
    "assumption below no longer holds"
)

human_pathopen.shape, human_pathvqa.shape, human_augmentation.shape


In [ ]:
def load_judge(model_key: str, dataset_filename: str, add_row_uid: bool = False) -> pd.DataFrame:
    path = os.path.join(JUDGE_OUTPUT_DIR, f"evaluator_{model_key}", dataset_filename)
    df = pd.read_csv(path)
    if add_row_uid:
        # The judge PathVQA CSV has no join key of its own: Image_ID repeats (one image
        # hosts several questions), and the runner does not emit row_uid. It IS built by
        # concatenating pathvqa_part1..part5 in sorted order, which is the same order
        # load_pooled_human() concatenates evaluator1..evaluator5 - so position is the key.
        #
        # Verified before relying on it: for both judges, all 662 rows match the pooled
        # human table positionally on BOTH Image_ID and OE_Question. (Compare NaN-safely -
        # 50 rows have a blank OE_Question, and NaN != NaN makes a naive comparison report
        # ~92% and look like a misalignment that isn't there.)
        df["row_uid"] = range(len(df))
    return df


judge_pathopen = {k: load_judge(k, "pathopen_eval_data.csv") for k in JUDGE_KEYS}
judge_pathvqa = {k: load_judge(k, "pathvqa_eval_data.csv", add_row_uid=True) for k in JUDGE_KEYS}
judge_augmentation = {
    k: load_judge(k, "pathopen_image_augmentation_eval_data.csv") for k in JUDGE_KEYS
}

# Fail loudly if that positional assumption ever breaks, rather than silently joining
# each judge score to the wrong question.
for _k, _judge_df in judge_pathvqa.items():
    assert len(_judge_df) == len(human_pathvqa), (
        f"{_k} PathVQA has {len(_judge_df)} rows vs {len(human_pathvqa)} pooled human rows; "
        "the positional row_uid join is no longer valid"
    )
    _h = human_pathvqa["Image_ID"].astype(str).values
    _j = _judge_df["Image_ID"].astype(str).values
    _matched = int((_h == _j).sum())
    assert _matched == len(_h), (
        f"{_k} PathVQA Image_ID aligns positionally on only {_matched}/{len(_h)} rows; "
        "the judge CSV is not in the same order as the pooled human table"
    )

# Augmentation joins on Image_ID (unique on both sides), so it needs no positional
# fallback - but check the overlap is total, since a partial join would silently shrink n.
for _k, _judge_df in judge_augmentation.items():
    _overlap = len(set(human_augmentation["Image_ID"]) & set(_judge_df["Image_ID"]))
    assert _overlap == len(human_augmentation), (
        f"{_k} augmentation shares only {_overlap}/{len(human_augmentation)} Image_IDs "
        "with the pooled human table"
    )

{k: v.shape for k, v in judge_pathopen.items()}, {k: v.shape for k, v in judge_augmentation.items()}


## Column mapping: which (benchmark, criterion) each pair of columns represents

Mirrors the rename step in `pathopen_scoring_analysis_individual.ipynb` /
`pathvqa_scoring_analysis_individual.ipynb`, but keeps a `(dataset, human_column,
judge_column, benchmark, criterion, question_type)` mapping explicit so both the
kappa and Mann-Whitney computations can iterate it directly.

In [ ]:
# Each entry: (question_type, human_column, judge_column, benchmark, criterion)
PATHOPEN_COLUMN_MAP = []
for i in (1, 2):
    PATHOPEN_COLUMN_MAP.append(("OE_correct", f"Evaluation OE_Correct_Answer_{i}\n(Benchmark 1)",
                                 f"Evaluation OE_Correct_Answer_{i}\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"))
    PATHOPEN_COLUMN_MAP.append(("OE_correct", f"Unnamed: {6 if i == 1 else 13}",
                                 f"OE_Correct_Answer_{i}_VisGround", 1, "Visual Grounding"))
    PATHOPEN_COLUMN_MAP.append(("OE_wrong", f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)",
                                 f"Evaluation OE_Wrong_Answer_{i}\n(Benchmark 2)", 2, "Error Proximity and Deductive Plausibility"))
    PATHOPEN_COLUMN_MAP.append(("OE_wrong", f"Unnamed: {9 if i == 1 else 16}",
                                 f"OE_Wrong_Answer_{i}_VisGroundErr", 2, "Visual Grounding Error"))

PATHOPEN_COLUMN_MAP.append(("MCQ_correct", "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)",
                             "Evaluation MCQ_OE_Correct_Answer\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"))
PATHOPEN_COLUMN_MAP.append(("MCQ_correct", "Unnamed: 20", "MCQ_OE_Correct_Answer_VisGround", 1, "Visual Grounding"))

for i in (1, 2, 3, 4):
    unnamed_idx = {1: 23, 2: 26, 3: 29, 4: 32}[i]
    PATHOPEN_COLUMN_MAP.append(("MCQ_wrong", f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)",
                                 f"Evaluation MCQ_OE_Wrong_Answer_{i}\n(Benchmark 2)", 2, "Error Proximity and Deductive Plausibility"))
    PATHOPEN_COLUMN_MAP.append(("MCQ_wrong", f"Unnamed: {unnamed_idx}",
                                 f"MCQ_OE_Wrong_Answer_{i}_VisGroundErr", 2, "Visual Grounding Error"))

PATHOPEN_COLUMN_MAP.append(("CE_correct", "Evaluation CE_Correct_Answer\n(Benchmark 3)",
                             "Evaluation CE_Correct_Answer\n(Benchmark 3)", 3, "Visual Grounding/Reasoning"))

PATHVQA_COLUMN_MAP = [
    ("OE_correct", "Evaluation OE_Correct_Answer\n(Benchmark 1)", "Evaluation OE_Correct_Answer\n(Benchmark 1)", 1, "Knowledge Interpretation/Deduction"),
    ("OE_correct", "Unnamed: 6", "OE_Correct_Answer_VisGround", 1, "Visual Grounding"),
    ("CE_correct", "Evaluation CE_Correct_Answer\n(Benchmark 3)", "Evaluation CE_Correct_Answer\n(Benchmark 3)", 3, "Visual Grounding/Reasoning"),
]

# Benchmark 4 - image augmentation (Table 8). Each augmentation row carries up to TWO
# question/answer pairs re-asked against the same augmented image, so there are two
# column blocks; pandas names the second one with a ".1" suffix.
#
# Column positions confirmed against the raw CSV's label row (index 0, dropped on load):
#   col 5 / 9  -> "Clinical Relevance"     (human: "Evaluation Image_Augmentation\n(Benchmark 4)[.1]")
#   col 6 / 10 -> "Visual Grounding"       (human: "Unnamed: 6" / "Unnamed: 10")
# The judge CSV names the Visual Grounding columns explicitly instead.
AUGMENTATION_COLUMN_MAP = [
    ("ImageAug_OE_1", "Evaluation Image_Augmentation\n(Benchmark 4)",
     "Evaluation Image_Augmentation\n(Benchmark 4)", 4, "Clinical Relevance"),
    ("ImageAug_OE_1", "Unnamed: 6",
     "OE_Image_Augmentation_1_VisGround", 4, "Visual Grounding"),
    ("ImageAug_OE_2", "Evaluation Image_Augmentation\n(Benchmark 4).1",
     "Evaluation Image_Augmentation\n(Benchmark 4).1", 4, "Clinical Relevance"),
    ("ImageAug_OE_2", "Unnamed: 10",
     "OE_Image_Augmentation_2_VisGround", 4, "Visual Grounding"),
]

len(PATHOPEN_COLUMN_MAP), len(PATHVQA_COLUMN_MAP), len(AUGMENTATION_COLUMN_MAP)


## Join judge scores to human scores

PathOPEN joins on `Image_ID` (unique across the pooled human dataset - verified
separately, no duplicates). Filtered PathVQA joins on the reconstructed `row_uid`
instead, since `Image_ID` repeats there (see note above).

Human and judge column names frequently collide (e.g. both call a column
`"Evaluation OE_Correct_Answer_1\n(Benchmark 1)"`), so after `merge(...,
suffixes=("_human", "_judge"))` pandas silently renames *both* copies of any
overlapping column - looking up the bare, unsuffixed name would raise a KeyError.
`_resolve_column` below always checks for the suffixed name first and only falls
back to the bare name when no collision occurred (e.g. judge-only columns like
`OE_Correct_Answer_1_VisGround`, which have no human counterpart with that exact
name and thus never sprout a suffix).

In [ ]:
VALID_SCORES = {-1, 0, 1, 2}
ORDERED_LEVELS = [-1, 0, 1, 2]   # fixed order so weight matrices are comparable across calls
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20260817


def _resolve_column(merged: pd.DataFrame, col: str, side: str) -> pd.Series:
    suffixed = f"{col}_{side}"
    if suffixed in merged.columns:
        return pd.to_numeric(merged[suffixed], errors="coerce")
    return pd.to_numeric(merged[col], errors="coerce")


def _quadratic_weights(levels=ORDERED_LEVELS) -> np.ndarray:
    """Quadratic agreement weights: 1 on the diagonal, falling off with squared distance.

    Matches sklearn's `weights="quadratic"` convention so AC1 and kappa penalise
    disagreements identically - the two statistics then differ ONLY in how they estimate
    chance agreement, which is the whole point of reporting both."""
    k = len(levels)
    span = (max(levels) - min(levels)) ** 2
    return np.array([[1 - ((levels[i] - levels[j]) ** 2) / span for j in range(k)]
                     for i in range(k)])


_WEIGHTS = _quadratic_weights()
_LEVEL_INDEX = {lvl: i for i, lvl in enumerate(ORDERED_LEVELS)}


def _to_index(values: np.ndarray) -> np.ndarray:
    """Map raw scores {-1,0,1,2} onto contiguous indices 0..3 once, so the bootstrap can
    work in index space and skip repeated label lookups."""
    out = np.empty(len(values), dtype=np.int64)
    for lvl, i in _LEVEL_INDEX.items():
        out[values == lvl] = i
    return out


def _confusion(ai: np.ndarray, bi: np.ndarray, k: int = len(ORDERED_LEVELS)) -> np.ndarray:
    """k x k confusion matrix via bincount - the hot path of the bootstrap.

    sklearn's cohen_kappa_score costs ~1.1 ms per call, which at 2000 resamples x ~200
    (criterion, judge) pairs x 2 statistics would be ~20 minutes. This is ~40x faster and
    is verified against sklearn below."""
    return np.bincount(ai * k + bi, minlength=k * k).reshape(k, k)


def _kappa_from_confusion(cm: np.ndarray) -> float:
    n = cm.sum()
    if n == 0:
        return np.nan
    observed = (_WEIGHTS * cm).sum() / n
    row, col = cm.sum(axis=1) / n, cm.sum(axis=0) / n
    expected = (_WEIGHTS * np.outer(row, col)).sum()
    if np.isclose(expected, 1.0):
        # Both raters constant on the same level: agreement is perfect but kappa is
        # undefined (0/0). Report 1.0 rather than NaN so the resample is not discarded,
        # which would bias the CI.
        return 1.0 if np.isclose(observed, 1.0) else np.nan
    return float((observed - expected) / (1 - expected))


def _ac1_from_confusion(cm: np.ndarray) -> float:
    n = cm.sum()
    if n == 0:
        return np.nan
    k = cm.shape[0]
    observed = (_WEIGHTS * cm).sum() / n
    pi = (cm.sum(axis=1) + cm.sum(axis=0)) / (2 * n)
    chance = (_WEIGHTS.sum() / (k * (k - 1))) * float((pi * (1 - pi)).sum())
    if np.isclose(chance, 1.0):
        return np.nan
    return float((observed - chance) / (1 - chance))


def gwet_ac1(a, b) -> float:
    """Gwet's AC1, quadratic-weighted, between two raters.

    Why this matters here: Cohen's kappa estimates chance agreement from the raters' own
    marginals, so when one category dominates - PathOPEN is ~95% twos - the chance term
    approaches the observed agreement and kappa collapses toward zero even though the
    raters agree on nearly every item. This is the well-known kappa paradox, and it is
    exactly the regime this dataset sits in.

    AC1 instead estimates chance agreement from how evenly the categories are used
    overall, which is stable under high prevalence. Reporting both is the honest move:
    kappa is the conventional number a reviewer expects, AC1 shows how much of a low
    kappa is prevalence artifact rather than genuine disagreement."""
    return _ac1_from_confusion(_confusion(_to_index(np.asarray(a)), _to_index(np.asarray(b))))


def weighted_kappa(a, b) -> float:
    return _kappa_from_confusion(_confusion(_to_index(np.asarray(a)), _to_index(np.asarray(b))))


def bootstrap_agreement_cis(a, b, n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED,
                            alpha: float = 0.05) -> dict:
    """Percentile bootstrap CIs for BOTH kappa and AC1 in one resampling pass.

    Resamples ITEMS (rating pairs) with replacement, not raters - the pairing between a
    human score and its judge score must survive resampling or the statistic is
    meaningless. Both statistics are computed from the same resample so their CIs
    describe the same uncertainty, and the expensive part (index shuffling) is done once."""
    a, b = np.asarray(a), np.asarray(b)
    n = len(a)
    if n < 3:
        return {k: np.nan for k in ("kappa_lo", "kappa_hi", "ac1_lo", "ac1_hi")}
    ai, bi = _to_index(a), _to_index(b)
    rng = np.random.default_rng(seed)
    kappas, ac1s = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        cm = _confusion(ai[idx], bi[idx])
        k_val, a_val = _kappa_from_confusion(cm), _ac1_from_confusion(cm)
        if np.isfinite(k_val):
            kappas.append(k_val)
        if np.isfinite(a_val):
            ac1s.append(a_val)
    floor = max(50, n_boot // 20)

    def _pct(values):
        if len(values) < floor:
            return np.nan, np.nan
        return (float(np.percentile(values, 100 * alpha / 2)),
                float(np.percentile(values, 100 * (1 - alpha / 2))))

    kappa_lo, kappa_hi = _pct(kappas)
    ac1_lo, ac1_hi = _pct(ac1s)
    return {"kappa_lo": kappa_lo, "kappa_hi": kappa_hi, "ac1_lo": ac1_lo, "ac1_hi": ac1_hi}


def compute_agreement(human_df: pd.DataFrame, judge_df: pd.DataFrame, column_map: list,
                      dataset_name: str, join_key: str, with_ci: bool = True) -> pd.DataFrame:
    """Weighted kappa AND Gwet's AC1, each with a bootstrapped 95% CI.

    -1 is a rubric level ("Unable to comprehend the question and/or image, or unable to
    make the evaluation"), not a missing value. A pathologist assigning it is making a
    substantive judgment that the ITEM is defective - which is exactly what Pillar 1's
    dataset-quality claim rests on. Excluding those rows would have removed the defective
    items from the agreement analysis and, worse, removed them asymmetrically: PathVQA has
    many more of them than PathOPEN, so the excluded rows were concentrated in the dataset
    the paper argues is lower quality.

    Whether the judge REPRODUCES the human's -1 is therefore a real validation question,
    and it can only be asked if -1 stays in. `n_human_neg1` / `n_judge_neg1` below expose
    how much of each kappa rests on those rows.

    Caveat to report alongside: quadratic weights treat the scale as evenly spaced, so a
    -1-vs-2 disagreement is penalized as a 3-step gap - heavier than 0-vs-2. -1 is
    arguably a different KIND of judgment ("unscorable") rather than a rung below 0, so
    these kappas are, if anything, conservative where raters disagree about scorability."""
    merged = human_df.merge(judge_df, on=join_key, suffixes=("_human", "_judge"))
    rows = []
    for question_type, human_col, judge_col, benchmark, criterion in column_map:
        if human_col not in human_df.columns or judge_col not in judge_df.columns:
            continue
        human_scores = _resolve_column(merged, human_col, "human")
        judge_scores = _resolve_column(merged, judge_col, "judge")

        # Keep every row where BOTH raters produced a value on the rubric scale.
        # Non-numeric / blank cells (-> NaN) are genuinely missing and still excluded.
        valid_mask = human_scores.isin(VALID_SCORES) & judge_scores.isin(VALID_SCORES)
        n = int(valid_mask.sum())
        human_valid = human_scores[valid_mask].astype(int).values
        judge_valid = judge_scores[valid_mask].astype(int).values

        if n < 2:
            kappa = ac1 = exact = np.nan
            cis = {"kappa_lo": np.nan, "kappa_hi": np.nan, "ac1_lo": np.nan, "ac1_hi": np.nan}
        else:
            cm = _confusion(_to_index(human_valid), _to_index(judge_valid))
            kappa = _kappa_from_confusion(cm)
            ac1 = _ac1_from_confusion(cm)
            exact = round(100 * float((human_valid == judge_valid).mean()), 1)
            cis = (bootstrap_agreement_cis(human_valid, judge_valid) if with_ci
                   else {"kappa_lo": np.nan, "kappa_hi": np.nan,
                         "ac1_lo": np.nan, "ac1_hi": np.nan})

        rows.append({
            "dataset": dataset_name,
            "question_type": question_type,
            "benchmark": benchmark,
            "criterion": criterion,
            "n": n,
            # How many of the n rows each rater called unscorable, and how often they
            # agreed on that call - a distinct signal from the kappa itself.
            "n_human_neg1": int((human_valid == -1).sum()) if n else 0,
            "n_judge_neg1": int((judge_valid == -1).sum()) if n else 0,
            "n_both_neg1": int(((human_valid == -1) & (judge_valid == -1)).sum()) if n else 0,
            "exact_agreement_pct": exact,
            "weighted_kappa": round(kappa, 4) if np.isfinite(kappa) else np.nan,
            "kappa_ci_low": round(cis["kappa_lo"], 4) if np.isfinite(cis["kappa_lo"]) else np.nan,
            "kappa_ci_high": round(cis["kappa_hi"], 4) if np.isfinite(cis["kappa_hi"]) else np.nan,
            "gwet_ac1": round(ac1, 4) if np.isfinite(ac1) else np.nan,
            "ac1_ci_low": round(cis["ac1_lo"], 4) if np.isfinite(cis["ac1_lo"]) else np.nan,
            "ac1_ci_high": round(cis["ac1_hi"], 4) if np.isfinite(cis["ac1_hi"]) else np.nan,
        })
    return pd.DataFrame(rows)


# The fast path must agree with sklearn exactly, or every kappa in the paper is suspect.
_rng = np.random.default_rng(7)
for _ in range(200):
    _x = _rng.choice(ORDERED_LEVELS, 60)
    _y = _rng.choice(ORDERED_LEVELS, 60)
    _mine = weighted_kappa(_x, _y)
    _theirs = cohen_kappa_score(_x, _y, weights="quadratic", labels=ORDERED_LEVELS)
    assert np.isclose(_mine, _theirs, atol=1e-10), f"{_mine} != {_theirs}"

# AC1 sanity: perfect agreement is 1, and it must exceed kappa under high prevalence.
_perfect = np.array([2, 2, 2, 1, 0, -1])
assert np.isclose(gwet_ac1(_perfect, _perfect), 1.0), "AC1 of identical ratings must be 1"
_a, _b = np.array([2] * 49 + [1]), np.array([2] * 50)
_k, _ac1 = weighted_kappa(_a, _b), gwet_ac1(_a, _b)
assert _ac1 > _k, f"AC1 ({_ac1:.3f}) should exceed kappa ({_k:.3f}) under high prevalence"
print(f"kappa matches sklearn on 200 random cases; AC1 checks passed")
print(f"  prevalence demo: kappa={_k:.3f} vs AC1={_ac1:.3f}")


In [ ]:
agreement_tables = {}
for model_key in JUDGE_KEYS:
    pathopen_agreement = compute_agreement(human_pathopen, judge_pathopen[model_key], PATHOPEN_COLUMN_MAP, "PathOPEN", join_key="Image_ID")
    pathvqa_agreement = compute_agreement(human_pathvqa, judge_pathvqa[model_key], PATHVQA_COLUMN_MAP, "Filtered_PathVQA", join_key="row_uid")
    # Benchmark 4 / Pillar 5a. Kept as its own "dataset" label because the rated object is
    # an AUGMENTED image, not a PathOPEN question-answer pair - pooling it with the
    # PathOPEN rows would mix two different units of analysis.
    augmentation_agreement = compute_agreement(
        human_augmentation, judge_augmentation[model_key], AUGMENTATION_COLUMN_MAP,
        "PathOPEN_ImageAug", join_key="Image_ID",
    )
    combined = pd.concat(
        [pathopen_agreement, pathvqa_agreement, augmentation_agreement], ignore_index=True
    )
    combined["judge"] = model_key
    agreement_tables[model_key] = combined

all_agreement = pd.concat(agreement_tables.values(), ignore_index=True)
all_agreement.sort_values(["dataset", "benchmark", "criterion", "judge"])


In [ ]:
os.makedirs(os.path.join(os.getcwd(), "agreement_output"), exist_ok=True)
all_agreement.to_csv(os.path.join(os.getcwd(), "agreement_output", "judge_pathologist_weighted_kappa.csv"), index=False)


## Distribution comparison: PathOPEN vs. Filtered-PathVQA (Mann-Whitney U)

Mirrors Pillar 1a's stated method exactly: compare score distributions between
PathOPEN and filtered-PathVQA on the criteria both datasets share (Benchmark 1:
Knowledge Interpretation/Deduction + Visual Grounding on OE correct answers;
Benchmark 3: Visual Grounding/Reasoning on CE correct answers), computed separately
for the human ratings and for each judge, with rank-biserial correlation as the
effect size.

In [ ]:
def rank_biserial_from_u(u_stat: float, n1: int, n2: int) -> float:
    """Rank-biserial correlation effect size derived from the Mann-Whitney U statistic."""
    return 1 - (2 * u_stat) / (n1 * n2)


def bootstrap_rank_biserial_ci(a, b, n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED,
                               alpha: float = 0.05) -> tuple:
    """Percentile bootstrap CI for the rank-biserial effect size.

    §2.1 asks for "bootstrapped 95% confidence intervals reported per criterion" on the
    Pillar 1a comparison; this supplies them. The two groups are resampled INDEPENDENTLY
    (they are independent samples - different datasets - not paired observations)."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if len(a) < 3 or len(b) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n_boot):
        ra = rng.choice(a, size=len(a), replace=True)
        rb = rng.choice(b, size=len(b), replace=True)
        # A resample where every value is identical has no rank information; Mann-Whitney
        # is undefined there, so skip rather than record a spurious 0.
        if len(np.unique(np.concatenate([ra, rb]))) < 2:
            continue
        u, _ = mannwhitneyu(ra, rb, alternative="two-sided")
        estimates.append(rank_biserial_from_u(u, len(ra), len(rb)))
    if len(estimates) < max(50, n_boot // 20):
        return np.nan, np.nan
    return (float(np.percentile(estimates, 100 * alpha / 2)),
            float(np.percentile(estimates, 100 * (1 - alpha / 2))))


def compare_pathopen_vs_pathvqa(pathopen_scores: pd.Series, pathvqa_scores: pd.Series) -> dict:
    """Mann-Whitney U over the FULL ordinal scale {-1, 0, 1, 2}, with a bootstrapped
    95% CI on the rank-biserial effect size.

    -1 ("unable to comprehend / evaluate") is kept. It is the rubric's bottom level and a
    direct measure of item defectiveness, so it belongs in a comparison whose entire point
    is which dataset contains more defective items. Filtering it inverted that logic:
    PathVQA carries more -1s than PathOPEN, so excluding them removed PathVQA's worst
    items and understated exactly the gap the test is meant to detect.

    Only genuinely missing values (NaN) are dropped."""
    a = pathopen_scores.dropna()
    b = pathvqa_scores.dropna()
    if len(a) < 2 or len(b) < 2:
        return {"n_pathopen": len(a), "n_pathvqa": len(b), "n_neg1_pathopen": 0,
                "n_neg1_pathvqa": 0, "mean_pathopen": np.nan, "mean_pathvqa": np.nan,
                "u_stat": np.nan, "p_value": np.nan, "rank_biserial": np.nan,
                "rb_ci_low": np.nan, "rb_ci_high": np.nan}
    u_stat, p_value = mannwhitneyu(a, b, alternative="two-sided")
    effect = rank_biserial_from_u(u_stat, len(a), len(b))
    ci_low, ci_high = bootstrap_rank_biserial_ci(a.values, b.values)
    return {"n_pathopen": len(a), "n_pathvqa": len(b),
            # Reported so a distribution shifted by unscorable items is never confused
            # with one shifted by poor-but-scorable answers.
            "n_neg1_pathopen": int((a == -1).sum()), "n_neg1_pathvqa": int((b == -1).sum()),
            "mean_pathopen": round(a.mean(), 3), "mean_pathvqa": round(b.mean(), 3),
            "u_stat": u_stat, "p_value": p_value, "rank_biserial": round(effect, 4),
            "rb_ci_low": round(ci_low, 4) if np.isfinite(ci_low) else np.nan,
            "rb_ci_high": round(ci_high, 4) if np.isfinite(ci_high) else np.nan}


Human and judge CSVs use **different column names** for the same criterion
(the human CSV inherits pandas's `Unnamed: N` name for the paired Visual Grounding
score; the judge CSV names it explicitly, e.g. `OE_Correct_Answer_1_VisGround`), so
each (criterion, source) combination needs its own column name rather than one
shared name reused across human/judge frames.

In [ ]:
# label -> {source: (pathopen_col, pathvqa_col)}
shared_comparisons = {
    "OE_correct_KnowledgeInterpretation": {
        "human": ('Evaluation OE_Correct_Answer_1\n(Benchmark 1)', 'Evaluation OE_Correct_Answer\n(Benchmark 1)'),
        "judge": ('Evaluation OE_Correct_Answer_1\n(Benchmark 1)', 'Evaluation OE_Correct_Answer\n(Benchmark 1)'),
    },
    "OE_correct_VisualGrounding": {
        "human": ("Unnamed: 6", "Unnamed: 6"),
        "judge": ("OE_Correct_Answer_1_VisGround", "OE_Correct_Answer_VisGround"),
    },
    "CE_correct_VisualGrounding": {
        "human": ('Evaluation CE_Correct_Answer\n(Benchmark 3)', 'Evaluation CE_Correct_Answer\n(Benchmark 3)'),
        "judge": ('Evaluation CE_Correct_Answer\n(Benchmark 3)', 'Evaluation CE_Correct_Answer\n(Benchmark 3)'),
    },
}

mw_rows = []
for label, cols_by_source in shared_comparisons.items():
    human_pathopen_col, human_pathvqa_col = cols_by_source["human"]
    result = compare_pathopen_vs_pathvqa(
        pd.to_numeric(human_pathopen.get(human_pathopen_col), errors="coerce"),
        pd.to_numeric(human_pathvqa.get(human_pathvqa_col), errors="coerce"),
    )
    result.update({"criterion": label, "rater": "human"})
    mw_rows.append(result)

    judge_pathopen_col, judge_pathvqa_col = cols_by_source["judge"]
    for model_key in JUDGE_KEYS:
        result = compare_pathopen_vs_pathvqa(
            pd.to_numeric(judge_pathopen[model_key].get(judge_pathopen_col), errors="coerce"),
            pd.to_numeric(judge_pathvqa[model_key].get(judge_pathvqa_col), errors="coerce"),
        )
        result.update({"criterion": label, "rater": model_key})
        mw_rows.append(result)

mann_whitney_results = pd.DataFrame(mw_rows)
mann_whitney_results


In [ ]:
mann_whitney_results.to_csv(os.path.join(os.getcwd(), "agreement_output", "pathopen_vs_pathvqa_mannwhitney.csv"), index=False)


## Inter-rater consistency without overlapping assignments

§2.1 states that "inter-rater reliability across evaluators was computed via Fleiss' κ or
Krippendorff's α." **Neither statistic can be computed from this data**, and the paper
text needs to change rather than the number being manufactured.

Both require multiple raters on the **same items**. The assignment here is disjoint by
design - each evaluator received their own slice:

| dataset | items | overlap between evaluators |
|---|---|---|
| PathOPEN core | 231 | **0** |
| PathOPEN augmentation | 174 | **0** |
| Filtered PathVQA | 662 | **8 items** (incidental, not designed) |

The 8 PathVQA overlaps are an accident of the sampling, not a reliability sub-study, and
8 items cannot support a Fleiss' κ that anyone should believe.

### What *can* be shown

With disjoint assignments, the answerable question is not "do raters agree on this item?"
but **"do raters apply the rubric the same way?"** If five pathologists draw disjoint
samples from one population and produce statistically indistinguishable score
distributions, that is evidence of shared rubric interpretation - weaker than item-level
agreement, but real, and honestly framed.

Three complementary views, computed below:

1. **Per-evaluator score distributions** - the raw evidence, one row per
   (evaluator × benchmark × criterion). This is the "how many of each ordinal score did
   each evaluator grant" table.
2. **Chi-square homogeneity test** - are those distributions consistent with all five
   evaluators drawing from a common one? A non-significant result supports shared
   interpretation; a significant one localises which criterion the raters diverge on.
3. **Kruskal-Wallis on mean severity** - chi-square ignores that the scale is *ordered*
   (it would treat a 2→1 shift the same as 2→-1). Kruskal-Wallis tests specifically
   whether some evaluators are systematically harsher, which is the failure mode that
   matters for a rubric.

**This is not a substitute for Fleiss' κ and must not be reported as one.** Two raters
can produce identical marginal distributions while disagreeing on every individual item.
It bounds *systematic* rater bias, not item-level reliability. The honest framing for
§2.1: state that the disjoint design precludes classical IRR, report this instead, and
note that a designed overlap subset would be needed for a true κ.

In [ ]:
from scipy.stats import chi2_contingency, kruskal

# (pooled human frame, column map, dataset label) for every human-rated subset.
HUMAN_SUBSETS = [
    (human_pathopen, PATHOPEN_COLUMN_MAP, "PathOPEN"),
    (human_pathvqa, PATHVQA_COLUMN_MAP, "Filtered_PathVQA"),
    (human_augmentation, AUGMENTATION_COLUMN_MAP, "PathOPEN_ImageAug"),
]

MIN_PER_EVALUATOR = 10  # below this a per-evaluator distribution is too thin to test


def evaluator_scores(df: pd.DataFrame, column_map: list, benchmark: int, criterion: str) -> dict:
    """{evaluator: array of scores} for one (benchmark, criterion), pooled over every
    column block that measures it (e.g. OE_Correct_Answer_1 and _2)."""
    columns = [entry[1] for entry in column_map
               if entry[3] == benchmark and entry[4] == criterion and entry[1] in df.columns]
    if not columns:
        return {}
    out = {}
    for evaluator, group in df.groupby("__evaluator__"):
        values = pd.concat([pd.to_numeric(group[c], errors="coerce") for c in columns],
                           ignore_index=True).dropna()
        values = values[values.isin(VALID_SCORES)].astype(int)
        if len(values):
            out[evaluator] = values.values
    return out


dist_rows, test_rows = [], []
for frame, column_map, dataset in HUMAN_SUBSETS:
    combos = sorted({(entry[3], entry[4]) for entry in column_map})
    for benchmark, criterion in combos:
        by_evaluator = evaluator_scores(frame, column_map, benchmark, criterion)
        if len(by_evaluator) < 2:
            continue

        for evaluator, values in sorted(by_evaluator.items()):
            row = {"dataset": dataset, "benchmark": benchmark, "criterion": criterion,
                   "evaluator": evaluator, "n": len(values),
                   "mean": round(float(values.mean()), 3)}
            for level in ORDERED_LEVELS:
                row[f"n_{level}"] = int((values == level).sum())
                row[f"pct_{level}"] = round(100 * float((values == level).mean()), 1)
            dist_rows.append(row)

        usable = {e: v for e, v in by_evaluator.items() if len(v) >= MIN_PER_EVALUATOR}
        if len(usable) < 2:
            continue

        # Chi-square homogeneity: do all evaluators use the 4 score levels alike?
        # Drop all-zero columns - a level nobody used carries no information and would
        # make the expected-frequency table singular.
        table = np.array([[int((v == lvl).sum()) for lvl in ORDERED_LEVELS]
                          for v in usable.values()])
        table = table[:, table.sum(axis=0) > 0]
        chi2 = chi2_p = np.nan
        expected_below_5 = np.nan
        if table.shape[1] >= 2:
            chi2, chi2_p, _, expected = chi2_contingency(table)
            # Chi-square is unreliable when expected counts are small; report the share
            # rather than silently trusting the p-value.
            expected_below_5 = round(100 * float((expected < 5).mean()), 1)

        # Kruskal-Wallis on the ordinal scores: is any evaluator systematically harsher?
        kw_h = kw_p = np.nan
        if len({tuple(v) for v in usable.values()}) > 1:
            try:
                kw_h, kw_p = kruskal(*usable.values())
            except ValueError:
                pass  # every value identical across all evaluators -> no variance to test

        means = {e: float(v.mean()) for e, v in usable.items()}
        test_rows.append({
            "dataset": dataset, "benchmark": benchmark, "criterion": criterion,
            "n_evaluators": len(usable), "n_total": int(sum(len(v) for v in usable.values())),
            "mean_spread": round(max(means.values()) - min(means.values()), 3),
            "harshest": min(means, key=means.get), "most_lenient": max(means, key=means.get),
            "chi2": round(chi2, 3) if np.isfinite(chi2) else np.nan,
            "chi2_p": f"{chi2_p:.4g}" if np.isfinite(chi2_p) else "n/a",
            "chi2_expected_lt5_pct": expected_below_5,
            "kruskal_H": round(kw_h, 3) if np.isfinite(kw_h) else np.nan,
            "kruskal_p": f"{kw_p:.4g}" if np.isfinite(kw_p) else "n/a",
            "distributions_differ_0.05": bool(np.isfinite(chi2_p) and chi2_p < 0.05),
        })

evaluator_distributions = pd.DataFrame(dist_rows)
evaluator_homogeneity = pd.DataFrame(test_rows)

print("===== per-evaluator score distributions (first 12 rows) =====")
print(evaluator_distributions.head(12).to_string(index=False))
print(f"\n({len(evaluator_distributions)} rows total)")
print("\n===== homogeneity across evaluators =====")
print(evaluator_homogeneity.to_string(index=False))


In [ ]:
evaluator_distributions.to_csv(
    os.path.join(os.getcwd(), "agreement_output", "evaluator_score_distributions.csv"),
    index=False)
evaluator_homogeneity.to_csv(
    os.path.join(os.getcwd(), "agreement_output", "evaluator_homogeneity_tests.csv"),
    index=False)

n_differ = int(evaluator_homogeneity["distributions_differ_0.05"].sum())
n_tested = len(evaluator_homogeneity)
print(f"criteria where evaluator distributions differ significantly (chi-square, p<0.05): "
      f"{n_differ}/{n_tested}")
if n_differ:
    print("\nthese criteria show systematic rater divergence and should be reported "
          "as such rather than pooled silently:")
    print(evaluator_homogeneity[evaluator_homogeneity["distributions_differ_0.05"]][
        ["dataset", "benchmark", "criterion", "mean_spread", "harshest", "most_lenient",
         "chi2_p", "kruskal_p"]].to_string(index=False))


## Benchmark 4 / Pillar 5a: augmentation score distributions (Figure 10 Panel A)

The paper's Figure 10 Panel A asks for "Score distribution, Clinical Relevance and
Visual Grounding, original vs. augmented."

**The literal comparison it names is not available**, and this is worth stating in the
Methods rather than papering over: Benchmark 4 was only ever applied to *augmented*
images (§3.5.3 - the pathologist sees an augmented image with the original question and
correct answer). There is no Benchmark 4 rating of an original image to put on the other
side of an "original vs. augmented" chart, and the two Benchmark 4 criteria (Clinical
Relevance, Visual Grounding-of-the-image) do not exist anywhere else in the rubric set.

Two defensible readings, both computed below:

1. **Absolute augmented-image quality** - the distribution of Benchmark 4 scores on their
   own. This is what actually validates the claim "augmentation preserves diagnostic
   content": if ~88% of augmented images score 2 for Clinical Relevance, the pipeline is
   not destroying the diagnosis. This is the number Table 4 already reports.
2. **Augmented vs. original, via the closest comparable criterion** - Benchmark 4's
   *Visual Grounding* against Benchmark 1's *Visual Grounding* on the same cases' original
   images. Both ask "does the image carry the information the answer needs?", so this is
   the nearest like-for-like contrast. It is an approximation and should be labelled as
   one - the two rubrics word the criterion differently (Table 5 vs. Table 8).

Panel B ("breakdown by transform type") is **not computable from the current data**: the
checkpoint records `Image_ID` but not which of the 16 transforms were sampled for that
image, so there is nothing to group by. Producing it would require the augmentation
pipeline to emit a per-image transform manifest.

In [ ]:
SCORE_LEVELS = [2, 1, 0, -1]


def _stack_criterion(df: pd.DataFrame, column_map: list, criterion: str, side: str) -> pd.Series:
    """All scores for one criterion, pooled across that criterion's column blocks.

    Benchmark 4 has two question blocks per augmented image (OE_1, OE_2); a row may
    carry a rating for one or both. Pooling them is correct here - the unit of analysis
    is the (image, question) rating, matching how Table 4's percentages were computed."""
    idx = 1 if side == "human" else 2
    parts = [pd.to_numeric(df[entry[idx]], errors="coerce")
             for entry in column_map
             if entry[4] == criterion and entry[idx] in df.columns]
    return pd.concat(parts, ignore_index=True).dropna() if parts else pd.Series(dtype=float)


def score_distribution(scores: pd.Series, label: str, rater: str, criterion: str) -> dict:
    scores = scores[scores.isin(SCORE_LEVELS)]
    n = len(scores)
    row = {"subset": label, "rater": rater, "criterion": criterion, "n": n}
    for level in SCORE_LEVELS:
        row[f"pct_{level}"] = round(100 * (scores == level).sum() / n, 1) if n else np.nan
    row["mean"] = round(scores.mean(), 3) if n else np.nan
    return row


dist_rows = []
for criterion in ["Clinical Relevance", "Visual Grounding"]:
    dist_rows.append(score_distribution(
        _stack_criterion(human_augmentation, AUGMENTATION_COLUMN_MAP, criterion, "human"),
        "PathOPEN_ImageAug", "human", criterion))
    for model_key in JUDGE_KEYS:
        dist_rows.append(score_distribution(
            _stack_criterion(judge_augmentation[model_key], AUGMENTATION_COLUMN_MAP, criterion, "judge"),
            "PathOPEN_ImageAug", model_key, criterion))

# Reading 2: the original-image comparator. Benchmark 1's Visual Grounding on PathOPEN OE
# correct answers is the nearest equivalent criterion rated on ORIGINAL images.
_b1_vg = [e for e in PATHOPEN_COLUMN_MAP if e[4] == "Visual Grounding" and e[0] == "OE_correct"]
dist_rows.append(score_distribution(
    _stack_criterion(human_pathopen, _b1_vg, "Visual Grounding", "human"),
    "PathOPEN_original (B1 VisGround)", "human", "Visual Grounding"))
for model_key in JUDGE_KEYS:
    dist_rows.append(score_distribution(
        _stack_criterion(judge_pathopen[model_key], _b1_vg, "Visual Grounding", "judge"),
        "PathOPEN_original (B1 VisGround)", model_key, "Visual Grounding"))

augmentation_distribution = pd.DataFrame(dist_rows)
print(augmentation_distribution.to_string(index=False))
augmentation_distribution


In [ ]:
augmentation_distribution.to_csv(
    os.path.join(os.getcwd(), "agreement_output", "benchmark4_augmentation_distribution.csv"),
    index=False)

# Augmented vs. original on the nearest-equivalent criterion (see caveat above).
aug_vs_original = []
for rater in ["human"] + JUDGE_KEYS:
    if rater == "human":
        aug = _stack_criterion(human_augmentation, AUGMENTATION_COLUMN_MAP, "Visual Grounding", "human")
        orig = _stack_criterion(human_pathopen, _b1_vg, "Visual Grounding", "human")
    else:
        aug = _stack_criterion(judge_augmentation[rater], AUGMENTATION_COLUMN_MAP, "Visual Grounding", "judge")
        orig = _stack_criterion(judge_pathopen[rater], _b1_vg, "Visual Grounding", "judge")
    aug, orig = aug[aug.isin(SCORE_LEVELS)], orig[orig.isin(SCORE_LEVELS)]
    if len(aug) < 2 or len(orig) < 2:
        continue
    # Independent samples, not paired: the augmentation subset (36 cases/pathologist) is
    # not the same row set as the full PathOPEN core subset, so a paired test would be wrong.
    u_stat, p_value = mannwhitneyu(orig, aug, alternative="two-sided")
    aug_vs_original.append({
        "rater": rater, "criterion": "Visual Grounding (B1 original vs B4 augmented)",
        "n_original": len(orig), "n_augmented": len(aug),
        "mean_original": round(orig.mean(), 3), "mean_augmented": round(aug.mean(), 3),
        "u_stat": u_stat, "p_value": p_value,
        "rank_biserial": round(rank_biserial_from_u(u_stat, len(orig), len(aug)), 3),
    })

aug_vs_original_df = pd.DataFrame(aug_vs_original)
if len(aug_vs_original_df):
    print(aug_vs_original_df.to_string(index=False))
    print("\nNegative rank_biserial => ORIGINAL images rank higher than augmented.")
    aug_vs_original_df.to_csv(
        os.path.join(os.getcwd(), "agreement_output", "benchmark4_augmented_vs_original.csv"),
        index=False)
aug_vs_original_df


## Summary

`agreement_output/judge_pathologist_weighted_kappa.csv` reports, per (dataset,
benchmark, criterion, judge), the weighted Cohen's kappa between that judge and
whichever single pathologist rated each row, alongside n and the `-1` breakdown
(`n_human_neg1`, `n_judge_neg1`, `n_both_neg1`). Report these honestly even if only
moderate, per the paper's own stated reviewer concern.

`agreement_output/pathopen_vs_pathvqa_mannwhitney.csv` reports the Mann-Whitney U /
rank-biserial effect size for PathOPEN-vs-filtered-PathVQA score distributions, for
human ratings and each judge separately - this lets you see whether a judge
reproduces the *same qualitative conclusion* (PathOPEN scores higher) as the human
evaluators did, independent of raw kappa agreement on individual rows.

Both tables now compute over the full `{-1, 0, 1, 2}` scale (see the `-1` section at
the top). The `n_*_neg1` columns are worth reporting in their own right: the rate at
which pathologists marked items unscorable is a per-dataset quality statistic, and
the rate at which a judge reproduces those calls is a per-judge capability statistic.

Two things to check when reading the regenerated numbers:

- **`n` rises** for every criterion where the human used `-1`, since those rows are no
  longer dropped. A kappa that moved should be attributed to the added rows, not to
  any change in the scores themselves - the underlying CSVs are untouched.
- **Kappa can move in either direction.** It rises where the judge reproduces the
  human's `-1` and falls where it does not, and under quadratic weights the
  disagreements it introduces are the heavily-penalized kind. A drop here is a real
  finding about judge reliability on defective items, not a regression.
